This notebook performs read-only exploratory checks of the Ariel ADC 2023
training and challenge-test datasets.

Its purpose is to establish the actual file structure, sample counts, schemas,
identifier alignment and spectral dimensions required for the Track A data
pipeline.

This notebook does not create data splits, fit preprocessing transformations,
train models or perform conformal calibration.

In [1]:
from pathlib import Path

import h5py
import numpy as np
import pandas as pd

TRAIN_DIR = Path("../FullDataset/TrainingData")
TEST_DIR = Path("../FullDataset/TestData")
GROUND_TRUTH_DIR = TRAIN_DIR / "Ground Truth Package"

TRAIN_AUX_PATH = TRAIN_DIR / "AuxillaryTable.csv"
TRAIN_SPECTRA_PATH = TRAIN_DIR / "SpectralData.hdf5"
TARGET_PATH = GROUND_TRUTH_DIR / "FM_Parameter_Table.csv"

TEST_AUX_PATH = TEST_DIR / "AuxillaryTable.csv"
TEST_SPECTRA_PATH = TEST_DIR / "SpectralData.hdf5"

required_paths = [
    TRAIN_AUX_PATH,
    TRAIN_SPECTRA_PATH,
    TARGET_PATH,
    TEST_AUX_PATH,
    TEST_SPECTRA_PATH,
]

for path in required_paths:
    print(f"{path}: {'FOUND' if path.exists() else 'MISSING'}")

../FullDataset/TrainingData/AuxillaryTable.csv: FOUND
../FullDataset/TrainingData/SpectralData.hdf5: FOUND
../FullDataset/TrainingData/Ground Truth Package/FM_Parameter_Table.csv: FOUND
../FullDataset/TestData/AuxillaryTable.csv: FOUND
../FullDataset/TestData/SpectralData.hdf5: FOUND


Loads the training metadata, simulator targets and test
metadata into pandas DataFrames.

For each table, it provides:

- the number of rows and columns
- the available column names
- the number of duplicated planet identifiers
- the total number of missing cells

These checks establish the available metadata and target fields and provide an
initial integrity check. Unique planet identifiers are required because the
future pipeline will align spectra, metadata and targets by `planet_ID`, rather
than assuming that separate files have the same row order.

In [2]:
train_aux = pd.read_csv(TRAIN_AUX_PATH)
targets = pd.read_csv(TARGET_PATH)
test_aux = pd.read_csv(TEST_AUX_PATH)

tables = {
    "training auxiliary": train_aux,
    "simulator targets": targets,
    "external test auxiliary": test_aux,
}

for name, table in tables.items():
    duplicate_count = int(table["planet_ID"].duplicated().sum())
    missing_count = int(table.isna().sum().sum())

    print(f"\n{name}")
    print("shape:", table.shape)
    print("columns:", table.columns.tolist())
    print("duplicate planet IDs:", duplicate_count)
    print("missing values:", missing_count)


training auxiliary
shape: (41423, 9)
columns: ['planet_ID', 'star_distance', 'star_mass_kg', 'star_radius_m', 'star_temperature', 'planet_mass_kg', 'planet_orbital_period', 'planet_distance', 'planet_surface_gravity']
duplicate planet IDs: 0
missing values: 0

simulator targets
shape: (41423, 9)
columns: ['Unnamed: 0', 'planet_ID', 'planet_radius', 'planet_temp', 'log_H2O', 'log_CO2', 'log_CO', 'log_CH4', 'log_NH3']
duplicate planet IDs: 0
missing values: 0

external test auxiliary
shape: (685, 9)
columns: ['planet_ID', 'star_distance', 'star_mass_kg', 'star_radius_m', 'star_temperature', 'planet_mass_kg', 'planet_orbital_period', 'planet_distance', 'planet_surface_gravity']
duplicate planet IDs: 0
missing values: 0


### Inspect representative spectral HDF5 groups

The spectral files use HDF5, where each planet is stored in a separate group
named `Planet_<planet_ID>`. This cell opens each file in read only mode and
inspects one representative training planet and one representative test planet.

It reports the total number of planet groups, the selected group's attributes,
and the name, shape and data type of each spectral dataset. This confirms the
expected per-planet structure without loading the entire spectral dataset into
memory.

In [3]:
def inspect_spectral_file(
    path: Path,
    example_planet_id: str,
) -> None:
    group_name = f"Planet_{example_planet_id}"

    with h5py.File(path, "r") as handle:
        group = handle[group_name]

        print(f"\nFile: {path}")
        print("planet groups:", len(handle))
        print("example group:", group_name)
        print("group attributes:", dict(group.attrs))

        for dataset_name, dataset in group.items():
            print(
                dataset_name,
                "shape=", dataset.shape,
                "dtype=", dataset.dtype,
            )


inspect_spectral_file(TRAIN_SPECTRA_PATH, "train1")
inspect_spectral_file(TEST_SPECTRA_PATH, "public1")


File: ../FullDataset/TrainingData/SpectralData.hdf5
planet groups: 41423
example group: Planet_train1
group attributes: {'ID': 'train1'}
instrument_noise shape= (52,) dtype= float64
instrument_spectrum shape= (52,) dtype= float64
instrument_width shape= (52,) dtype= float64
instrument_wlgrid shape= (52,) dtype= float64

File: ../FullDataset/TestData/SpectralData.hdf5
planet groups: 685
example group: Planet_public1
group attributes: {'ID': 'public1'}
instrument_noise shape= (52,) dtype= float64
instrument_spectrum shape= (52,) dtype= float64
instrument_width shape= (52,) dtype= float64
instrument_wlgrid shape= (52,) dtype= float64


# NOTE:

The test dataset is separate from the labelled data used for
the train, validation, calibration and evaluation partitions. It
contains spectra and metadata but no simulator targets, so it cannot be used
to evaluate prediction accuracy or conformal coverage.

### Verify identifier alignment across source files

Spectra, auxiliary metadata and simulator targets are stored separately. They
must be joined using the explicit `planet_ID` field rather than their
row positions.

For spectral HDF5 files, the planet identifier is encoded in each root group
name as `Planet_<planet_ID>`. This check extracts those identifiers and compares
their sets with the corresponding CSV identifiers.

Matching identifier sets mean that each source contains the same planets, even
if its rows or groups appear in a different order. The future loader will still
perform an explicit identifier-based join - this does not make row-order
alignment acceptable.

In [4]:
train_aux

,planet_ID,star_distance,star_mass_kg,star_radius_m,star_temperature,planet_mass_kg,planet_orbital_period,planet_distance,planet_surface_gravity
0,train1,530.7650,1.968526e+30,6.678720e+08,5636.0,3.990923e+26,9.668577,0.088530,16.642039
1,train2,440.0890,1.948642e+30,7.235280e+08,5449.0,4.977001e+26,4.218028,0.050752,5.197135
2,train3,1126.3700,2.147483e+30,7.791840e+08,5675.0,9.686804e+25,2.828042,0.040158,7.871718
3,train4,1257.1600,2.048062e+30,7.374420e+08,5849.0,4.486340e+25,3.855604,0.048601,4.894681
4,train5,211.5530,2.525281e+30,1.092249e+09,6200.0,2.416329e+26,3.076340,0.044833,5.870151
...,...,...,...,...,...,...,...,...,...
41418,train41419,703.3020,1.988410e+30,7.722270e+08,5821.0,7.125924e+26,5.370647,0.060024,4.544766
41419,train41420,72.7998,1.292466e+30,4.035060e+08,4258.0,7.709155e+25,6.342000,0.058089,13.050003
41420,train41421,1786.4800,1.789569e+30,7.096140e+08,6002.0,2.794853e+25,3.546511,0.043946,19.728779
41421,train41422,329.0410,1.849221e+30,6.748290e+08,5519.0,1.400242e+26,22.631964,0.152856,3.649544


In [5]:
targets

,Unnamed: 0,planet_ID,planet_radius,planet_temp,log_H2O,log_CO2,log_CO,log_CH4,log_NH3
0,0,train1,0.559620,863.394770,-8.865868,-6.700707,-5.557561,-8.957615,-3.097540
1,1,train2,1.118308,1201.700465,-4.510258,-8.228966,-3.565427,-7.807424,-3.633658
2,2,train3,0.400881,1556.096477,-7.225472,-6.931472,-3.081975,-8.567854,-5.378472
3,3,train4,0.345974,1268.624884,-7.461157,-5.853334,-3.044711,-5.149378,-3.815568
4,4,train5,0.733184,1707.323564,-4.140844,-7.460278,-3.181793,-5.996593,-4.535345
...,...,...,...,...,...,...,...,...,...
41418,41418,train41419,1.430950,1167.563319,-7.556565,-7.075889,-4.735715,-7.532563,-7.854443
41419,41419,train41420,0.277752,649.703932,-7.434332,-7.161510,-5.274940,-4.100312,-3.651804
41420,41420,train41421,0.136016,1445.371723,-5.525906,-5.079240,-4.945121,-5.787630,-8.015444
41421,41421,train41422,0.707851,653.284686,-4.616695,-8.782546,-3.767229,-3.221403,-8.427805


In [6]:
test_aux

,planet_ID,star_distance,star_mass_kg,star_radius_m,star_temperature,planet_mass_kg,planet_orbital_period,planet_distance,planet_surface_gravity
0,public1,283.803,1.465591e+30,9.322380e+08,4803.0,2.509420e+27,3.550585,0.041144,32.771225
1,public2,223.465,2.230747e+30,1.426067e+09,6115.0,6.409426e+26,19.148095,0.145546,8.370251
2,public3,1516.960,3.313486e+30,1.690551e+09,7366.0,2.486766e+27,0.554233,0.015655,32.475379
3,public4,542.026,2.087383e+30,1.071378e+09,5999.1,6.496475e+26,3.007820,0.041445,8.483931
4,public5,597.937,1.804935e+30,8.835390e+08,5652.8,5.408769e+26,6.508121,0.066053,7.063465
...,...,...,...,...,...,...,...,...,...
680,public681,377.872,1.695438e+30,1.383462e+09,5441.0,3.663562e+26,10.530579,0.089158,7.307499
681,public682,366.987,2.666806e+30,1.252260e+09,6509.8,5.731968e+26,18.095475,0.148757,6.639195
682,public683,347.850,1.869105e+30,6.609150e+08,5295.0,6.680521e+26,4.301219,0.050708,5.655702
683,public684,124.680,1.533197e+30,5.693122e+08,5058.0,1.553480e+26,3.770130,0.043471,8.487737


In [7]:
targets[["Unnamed: 0", "planet_ID"]].head()

,Unnamed: 0,planet_ID
0,0,train1
1,1,train2
2,2,train3
3,3,train4
4,4,train5


In [8]:
print("Unnamed column unique:", targets["Unnamed: 0"].is_unique)
print("Unnamed column range:", targets["Unnamed: 0"].min(), targets["Unnamed: 0"].max())

Unnamed column unique: True
Unnamed column range: 0 41422


In [9]:
HDF5_GROUP_PREFIX = "Planet_"

def extract_spectral_ids(path: Path) -> set[str]:
    """Extract planet IDs from the root groups of a spectral HDF5 file."""
    with h5py.File(path, "r") as handle:
        group_names = list(handle.keys())

        unexpected_names = [
            name
            for name in group_names
            if not name.startswith(HDF5_GROUP_PREFIX)
        ]

    if unexpected_names:
        raise ValueError(
            f"{path} containts unexpected root group names:"
            f"{unexpected_names[:5]}"
        )

    return {name.removeprefix(HDF5_GROUP_PREFIX) for name in group_names}


def extract_csv_ids(table: pd.DataFrame, source_name: str,) -> set[str]:
    """Validate and extract unique planet IDs from a CSV table."""

    if "planet_ID" not in table.columns:
        raise ValueError(f"{source_name} does not contain a planet_ID column")

    if table["planet_ID"].isna().any():
        raise ValueError(f"{source_name} contains missing planet IDs")

    duplicated = table.loc[table["planet_ID"].duplicated(), "planet_ID"].tolist()

    if duplicated:
        raise ValueError(f"{source_name} contains duplicate planet IDs: "
                         f"{duplicated[:5]}")

    return set(table["planet_ID"].astype(str))


def compare_id_sets(
        left_name: str,
        left_ids: set[str],
        right_name: str,
        right_ids: set[str],
) -> None:
    """Prints a concise comparison and fails if identifier sets different"""

    left_only = left_ids - right_ids
    right_only = right_ids - left_ids

    print(f"\n{left_name} vs {right_name}")
    print(f"{left_name}: {len(left_ids):,}")
    print(f"{right_name}: {len(right_ids):,}")
    print(f"Only in {left_name}: {len(left_only):,}")
    print(f"Only in {right_name}: {len(right_only):,}")

    if left_only or right_only:
        raise ValueError(
            "Identifier mismatch detected. "
            f"Examples only in {left_name}: "
            f"{sorted(left_only)[:5]}; "
            f"examples only in {right_name}: "
            f"{sorted(right_only)[:5]}"
        )

train_spectral_ids = extract_spectral_ids(TRAIN_SPECTRA_PATH)
challenge_spectral_ids = extract_spectral_ids(TEST_SPECTRA_PATH)

train_aux_ids = extract_csv_ids(
    train_aux,
    "training auxiliary table",
)
target_ids = extract_csv_ids(
    targets,
    "simulator target table",
)
challenge_aux_ids = extract_csv_ids(
    test_aux,
    "unlabelled challenge auxiliary table",
)

compare_id_sets(
    "training spectra",
    train_spectral_ids,
    "training auxiliary metadata",
    train_aux_ids,
)

compare_id_sets(
    "training auxiliary metadata",
    train_aux_ids,
    "simulator targets",
    target_ids,
)

compare_id_sets(
    "challenge spectra",
    challenge_spectral_ids,
    "challenge auxiliary metadata",
    challenge_aux_ids,
)

overlapping_ids = train_aux_ids & challenge_aux_ids

print(f"\nTraining/challenge ID overlap: {len(overlapping_ids):,}")

if overlapping_ids:
    raise ValueError(
        "Training and challenge datasets are not disjoint. "
        f"Overlapping examples: {sorted(overlapping_ids)[:5]}"
    )

print("\nAll identifier integrity checks passed.")


training spectra vs training auxiliary metadata
training spectra: 41,423
training auxiliary metadata: 41,423
Only in training spectra: 0
Only in training auxiliary metadata: 0

training auxiliary metadata vs simulator targets
training auxiliary metadata: 41,423
simulator targets: 41,423
Only in training auxiliary metadata: 0
Only in simulator targets: 0

challenge spectra vs challenge auxiliary metadata
challenge spectra: 685
challenge auxiliary metadata: 685
Only in challenge spectra: 0
Only in challenge auxiliary metadata: 0

Training/challenge ID overlap: 0

All identifier integrity checks passed.


## Explanation of above cell

`extract_spectral_ids`:
This opens an HDF5 file in read-only mode and reads its root group names.

For example:

`Planet_train1`

becomes:

`train1`


`extract_csv_ids`:
This checks that:
- `pkanet_ID` exists
- no ID is missing or duplicated
- Then converts the IDs to strings and returns a set to allow comparison of membership that is independent of row order.